# User Onboarding 및 Backend Wallet Operation

## 개요

Tutorial 00에서는 이후 agent 튜토리얼에서 사용할 수 있도록 developer용 wallet 하나를 생성했습니다. production에서는 애플리케이션의 *모든* 최종 사용자가 자체 embedded wallet을 가지며 backend가 사용자를 대신한 agent 지출을 제어하는 session budget을 관리합니다.

이 Notebook은 두 부분으로 구성됩니다.

- **Part 1 — Onboarding(최종 사용자별):** wallet 생성, 자금 입금, signing 위임, 선택적으로 추가 chain에 wallet provision
- **Part 2 — Backend operation:** balance 확인, budget이 있는 session 생성, instrument 목록 표시, 남은 budget 확인. 최종 사용자가 아니라 application backend에서 실행되며 전체 wallet lifecycle을 한 곳에서 볼 수 있도록 포함했습니다.

동일한 flow가 Coinbase CDP와 Stripe(Privy) wallet provider 모두에서 작동하며 서로 다른 부분에는 provider별 funding 및 delegation UI를 설명합니다.

## 사전 요구 사항

> **비용 안내:** 사용한 서비스의 비용에 관한 자세한 내용은 AgentCore pricing을 참조하세요 - https://aws.amazon.com/bedrock/agentcore/pricing/. 이 튜토리얼은 testnet resource를 사용하지만 AWS infrastructure에는 요금이 발생합니다. 작업을 마치면 Tutorial 00의 cleanup을 실행하세요.

* Tutorial 00 완료(`.env`에 `PAYMENT_MANAGER_ARN`, `PAYMENT_CONNECTOR_ID`, `LINKED_EMAIL` 존재)
* https://faucet.circle.com/의 testnet USDC(Base Sepolia 또는 Solana Devnet)

In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import sys
import os

sys.path.append("..")

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore.payments import PaymentManager
from utils import load_tutorial_env, print_summary, client_token, wait_for_status

config = load_tutorial_env()
PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]
NETWORK = os.environ.get("NETWORK", "ETHEREUM")

# instrument + session operation용 SDK client(flat하고 래핑되지 않은 dict 반환)
manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

# SDK에서 노출하지 않는 GetPaymentInstrumentBalance를 위해 boto3 client 유지
dp_client = boto3.client("bedrock-agentcore", region_name=REGION)

# connector ID 가져오기
if config.get("multi_provider"):
    PROVIDER = list(config["instruments"].keys())[0]
    CONNECTOR_ID = config["instruments"][PROVIDER]["connector_id"]
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
else:
    CONNECTOR_ID = config.get("connector_id")
    INSTRUMENT_ID = config.get("instrument_id")
    PROVIDER = config.get("provider_type", "unknown")

print_summary("Config", provider=PROVIDER, instrument=INSTRUMENT_ID)

> **이 튜토리얼에는 두 persona가 있습니다.** 아래 셀은 lifecycle의 양쪽을 시뮬레이션합니다. *application backend*는 wallet을 provision하고 balance 및 session operation을 실행하며, *최종 사용자*는 wallet에 자금을 입금하고 UI를 통해 동의합니다. 편의를 위해 developer email(`.env`의 `LINKED_EMAIL`)을 최종 사용자 identity로 재사용합니다. production에서는 각 사용자가 자신의 email과 wallet을 가집니다.


---

# Part 1 — Onboarding(최종 사용자별)

1~3섹션에서는 사용자가 처음 가입할 때 backend에서 wallet을 provision하고 사용자가 자금을 입금한 후 agent에 sign 권한을 부여하는 과정을 다룹니다.


### Onboarding Flow

![Onboarding Flow](images/onboarding_flow.png)


## 1. Embedded Wallet 생성

핵심 onboarding 호출입니다. 사용자가 가입할 때마다 application backend가 `CreatePaymentInstrument`를 실행하여 사용자 identity(email, phone 또는 OAuth)에 연결된 embedded USDC wallet을 provision합니다. 사용자에게 private key는 노출되지 않으며 wallet provider(Coinbase 또는 Privy)가 사용자를 대신하여 key material을 보유합니다.

Tutorial 00에서는 developer wallet에 이 작업을 수행했습니다. 이 셀에서는 두 번째 사용자를 onboarding하여 end-to-end 호출을 확인합니다.

> **이 튜토리얼에서 Email 재사용:** 아래 코드 셀은 Tutorial 00에서 설정한 동일한 email인 `.env`의 `LINKED_EMAIL`을 읽으므로 "new user"는 실제로 다른 역할을 수행하는 본인입니다. 하나의 email을 사용하면 하나의 inbox, 한 세트의 OTP code, 하나의 wallet login만 사용하여 튜토리얼이 간단해집니다. 실제 signup flow에서는 여기에 새 사용자의 실제 email을 전달합니다.


In [ ]:
# 새 최종 사용자를 위한 두 번째 wallet provision
# 구성된 provider를 통해 wallet을 provision하는 실제 API 호출

# 새 wallet의 최종 사용자 identity. production에서는 각 사용자가
# 자체 email을 사용하지만 여기서는 튜토리얼을 간단하게 유지하도록
# .env의 developer LINKED_EMAIL을 재사용함. 상단의 두 persona 설명 참조
NEW_USER_ID = "tutorial-03-user"
NEW_EMAIL = os.environ.get("LINKED_EMAIL", "tutorial03@example.com")

# SDK가 instrument field를 flat하게 반환(paymentInstrumentId, paymentInstrumentDetails, status가
# top level에 있음)하므로 unwrapping 불필요
inst = manager.create_payment_instrument(
    user_id=NEW_USER_ID,
    payment_connector_id=CONNECTOR_ID,
    payment_instrument_type="EMBEDDED_CRYPTO_WALLET",
    payment_instrument_details={
        "embeddedCryptoWallet": {
            "network": NETWORK,
            "linkedAccounts": [{"email": {"emailAddress": NEW_EMAIL}}],
        }
    },
    client_token=client_token(),
)
NEW_INSTRUMENT_ID = inst["paymentInstrumentId"]
NEW_WALLET = inst["paymentInstrumentDetails"]["embeddedCryptoWallet"]["walletAddress"]

# Instrument는 일반적으로 즉시 ACTIVE가 됨. provisioning 중 API가 잠시
# INITIATED를 반환할 경우를 대비한 안전 확인
if inst.get("status") != "ACTIVE":
    print("\nWaiting for instrument to become ACTIVE...")
    wait_for_status(
        dp_client.get_payment_instrument,
        "ACTIVE",
        paymentManagerArn=PAYMENT_MANAGER_ARN,
        paymentConnectorId=CONNECTOR_ID,
        paymentInstrumentId=NEW_INSTRUMENT_ID,
        userId=NEW_USER_ID,
    )

print_summary(
    "New Instrument Created",
    instrument_id=NEW_INSTRUMENT_ID,
    wallet_address=NEW_WALLET,
    network=NETWORK,
    status="ACTIVE",
)

# Coinbase는 최종 사용자가 wallet에 자금을 입금하고 signing 권한을 부여하는
# entry point인 WalletHub의 redirectUrl을 제공하지만 Privy는 제공하지 않음
# 이에 해당하는 flow는 자체 frontend에 있으며 튜토리얼에서는 Privy reference frontend 사용(아래 2, 3섹션 참조)
redirect_url = inst["paymentInstrumentDetails"]["embeddedCryptoWallet"].get("redirectUrl")
if redirect_url:
    print(f"\n  WalletHub: {redirect_url}")
    print("  Share this URL with the end user to fund the wallet and grant signing permission.")

## 2. Wallet에 자금 입금

> **👤 최종 사용자 작업:** Funding은 최종 사용자가 UI(Coinbase는 WalletHub, Privy는 자체 frontend이며 튜토리얼에서는 Privy reference frontend 사용)를 통해 수행합니다. 이 튜토리얼에서는 developer가 최종 사용자 역할을 수행하여 아래 Circle faucet에서 wallet에 자금을 입금합니다.

### 튜토리얼용 Testnet Funding

Circle USDC faucet(무료 testnet token)을 사용합니다.
1. [faucet.circle.com](https://faucet.circle.com/)으로 이동합니다.
2. **Base Sepolia**(ETHEREUM) 또는 **Solana Devnet**(SOLANA)을 선택합니다.
3. wallet address(위 **New Instrument Created** 요약의 `Wallet Address`)를 붙여넣고 USDC를 요청합니다.

### Funding Flow(참고용)

애플리케이션에서 최종 사용자는 provider별 UI를 통해 wallet에 자금을 입금합니다. 사용자가 경험할 내용을 이해할 수 있도록 살펴보세요. 튜토리얼 테스트에는 위 faucet을 계속 사용합니다.

#### Coinbase CDP — WalletHub

Coinbase instrument를 생성하면 response에 Coinbase WalletHub를 가리키는 `redirectUrl`이 포함됩니다. 이 URL을 최종 사용자와 공유하면 동일한 UI에서 다음 작업을 수행할 수 있습니다.

* wallet 자금 입금(crypto transfer 또는 credit card / bank / Apple Pay / Google Pay를 통한 fiat onramp)
* agent에 signing 권한 부여(delegation — 3섹션 참조)

WalletHub는 Coinbase에서 전적으로 관리하고 Coinbase domain에 호스팅하므로 직접 배포할 항목이 없습니다.

#### Stripe(Privy) — Privy Reference Frontend

Tutorial 00의 3단계에서 `http://localhost:3000`에 Privy reference frontend를 시작했습니다. 동일한 UI의 **Add funds** 작업에서 세 가지 option을 제공합니다.

* **Pay with card** — Stripe onramp를 통한 fiat → USDC(KYC, payment processing, delivery 처리)
* **Transfer from wallet** — 사용자가 이미 제어하는 외부 wallet에서 crypto-to-crypto 전송
* **Receive funds** — 다른 사람이 USDC를 보낼 수 있도록 wallet address와 QR code 표시

![Privy funding option](images/03-privy-fund-options.png)

> **참고:** `http://localhost:3000`은 튜토리얼 전용입니다. production에서는 실제 HTTPS domain에 Privy reference frontend 또는 자체 구현을 배포하고 Privy flow를 애플리케이션에 직접 통합합니다.

### Fiat-to-Crypto Onramp 용어

**Fiat**은 credit card, bank transfer, Apple Pay 또는 Google Pay로 결제하는 기존 통화(USD, EUR)를 의미합니다. **onramp**는 fiat을 stablecoin(USDC)으로 변환하여 embedded wallet에 입금합니다. 튜토리얼 실행에서는 faucet의 testnet USDC를 사용하며 실제 자금을 사용하지 않습니다.

| Provider | Onramp 문서 |
|----------|-------------|
| Coinbase | [Coinbase Onramp](https://docs.cdp.coinbase.com/onramp/coinbase-hosted-onramp/generating-onramp-url) · [Sandbox Testing](https://docs.cdp.coinbase.com/onramp/additional-resources/sandbox-testing) |
| Stripe (Privy) | [Stripe Onramp](https://docs.stripe.com/crypto/onramp) |



## 3. Delegation: Signing 권한 부여

> ✋ **수동 단계:** Delegation은 이 Notebook 외부에서 수행됩니다. developer가 mechanism을 설정하고 최종 사용자가 frontend(튜토리얼에서는 Privy reference frontend 사용) 또는 WalletHub를 통해 동의합니다.

agent가 transaction에 sign하기 전에 최종 사용자가 권한을 부여합니다. wallet마다 한 번 수행하는 단계입니다.

| | Coinbase CDP | Stripe (Privy) |
|---|---|---|
| **Mechanism** | project-level delegated signing | wallet의 additional signer인 authorization key |
| **설정** | CDP Portal → Wallets → Embedded Wallet → Policies → enable | Privy reference frontend `addSigners()` 호출 |
| **사용자 작업** | WalletHub `redirectUrl`을 통해 동의 | 1. 최종 사용자 email로 `http://localhost:3000`에 로그인
2. **Connect agent** 선택
3. **Give access** 선택

2섹션의 Add funds와 동일한 UI |
| **범위** | project 아래의 모든 wallet | wallet별 |
| **미설정 시** | ProcessPayment 오류 발생 | ProcessPayment가 HTTP 500 반환 |

> **Production 참고 사항:** `http://localhost:3000`은 튜토리얼 테스트 전용입니다. production에서는 실제 HTTPS domain에서 Privy reference frontend 또는 자체 구현을 실행하고 Privy flow를 애플리케이션에 포함합니다. Coinbase WalletHub는 Coinbase domain의 동일한 managed UI에서 fund와 delegate를 모두 처리하므로 직접 호스팅할 필요가 없습니다.

### Wallet Provider Path

![Wallet Provider Path](images/wallet_providers.png)


---

# Part 2 — Backend Operation

나머지 섹션은 최종 사용자가 아니라 application backend에서 실행합니다. agent task 전후 및 실행 중에 호출하는 operation인 balance 확인, 추가 chain에 wallet 추가, session budget, instrument 목록, 남은 budget query를 다룹니다. persona 경계를 명확히 알 수 있도록 각 섹션에 🖥️ 표시가 있습니다.

아래 셀은 1섹션에서 onboarding한 사용자를 재사용하므로 실제 wallet을 대상으로 호출합니다.


## 4. Wallet Balance 확인

> **🖥️ Backend operation:** application backend가 `GetPaymentInstrumentBalance`를 호출합니다. 최종 사용자는 이 API를 직접 호출하지 않으며 WalletHub 또는 자체 UI에서 balance를 확인합니다.

session을 생성하기 전에 wallet에 USDC가 있는지 검증합니다.

이 Notebook의 나머지 부분에서는 AgentCore SDK의 `PaymentManager`를 사용하지만 `GetPaymentInstrumentBalance`는 SDK를 통해 노출되지 않으므로 이 셀에서는 boto3 `dp_client`를 직접 사용합니다. Tutorial 00도 동일한 방식을 사용합니다.

In [ ]:
# instrument balance 확인
chain = "BASE_SEPOLIA" if NETWORK == "ETHEREUM" else "SOLANA_DEVNET"

# boto3 model에서는 optional로 표시되지만 service는 이 호출에 userId를 요구함
# 각 wallet은 자체 userId로 생성되었으므로 (instrument_id, user_id)를 함께 연결
for label, inst_id, user_id in [
    ("Tutorial 00 instrument", INSTRUMENT_ID, USER_ID),
    ("New instrument", NEW_INSTRUMENT_ID, NEW_USER_ID),
]:
    try:
        resp = dp_client.get_payment_instrument_balance(
            paymentManagerArn=PAYMENT_MANAGER_ARN,
            paymentConnectorId=CONNECTOR_ID,
            paymentInstrumentId=inst_id,
            userId=user_id,
            chain=chain,
            token="USDC",
        )
        balance = resp.get("tokenBalance", {})
        amount = int(balance.get("amount", "0")) / 1_000_000
        print(f"✅ {label}: {amount:.2f} USDC on {chain}")
        if amount == 0:
            print("   Fund at: https://faucet.circle.com/")
    except Exception as e:
        print(f"⚠️  {label}: {e}")

## 5. Multi-Network Wallet

> **🖥️ Backend operation:** backend가 `CreatePaymentInstrument`를 두 번째로 호출하여 다른 chain에 wallet을 추가합니다. 사용자가 실행하는 작업이 아니며 동일한 identity 아래에 추가 wallet을 받습니다.

동일한 사용자가 같은 PaymentManager 아래에서 Ethereum 및 Solana wallet을 모두 가질 수 있습니다. 서로 다른 `network` 값으로 `create_payment_instrument`를 두 번 호출합니다. provider가 기존 사용자를 감지하고 요청한 chain에 새 wallet을 추가합니다.

```python
# Ethereum wallet(Tutorial 00에서 이미 생성)
eth_instrument = manager.create_payment_instrument(
    ..., payment_instrument_details={'embeddedCryptoWallet': {'network': 'ETHEREUM', ...}}
)

# Solana wallet(동일한 user, manager, connector)
sol_instrument = manager.create_payment_instrument(
    ..., payment_instrument_details={'embeddedCryptoWallet': {'network': 'SOLANA', ...}}
)
```

각 instrument에는 자체 wallet address가 있으며 개별적으로 자금을 입금해야 합니다. agent는 paid endpoint의 network와 일치하는 `instrumentId`를 받습니다.

## 6. 서로 다른 Payment Limit으로 Session 생성

> **🖥️ Backend operation:** `CreatePaymentSession`은 backend 호출입니다. application이 agent에 작업을 routing하기 직전에 task별 session을 provision합니다. 최종 사용자는 session을 직접 생성하지 않습니다.

실제로 backend는 모든 작업에 하나의 session을 사용하는 대신 task별로 새 session을 생성합니다. 서로 다른 task에는 서로 다른 budget과 expiry time을 적용합니다.

> **wallet은 어디에 있나요?** `CreatePaymentSession`은 instrument가 아니라 사용자와 budget만 받습니다. Session은 wallet을 인식하지 않습니다. `ProcessPayment` 시 service가 merchant의 x402 challenge와 network가 일치하는 사용자 instrument를 선택합니다. 개별 payment가 budget에 맞는 한 단일 session에서 사용자의 Ethereum 및 Solana wallet 전체에 걸쳐 지출할 수 있습니다.

In [ ]:
# SDK가 session field를 flat하게 반환(paymentSessionId, limits, expiryTimeInMinutes가
# top level에 있음)하므로 `quick`, `research`, `deep`은 바로 사용할 수 있는 dict임

# 빠른 조회: 작은 budget, 짧은 expiry
quick = manager.create_payment_session(
    user_id=USER_ID,
    expiry_time_in_minutes=15,
    limits={"maxSpendAmount": {"value": "0.10", "currency": "USD"}},
    client_token=client_token(),
)
print(f"Quick lookup: {quick['paymentSessionId']} ($0.10 / 15 min)")

# research task: 중간 budget
research = manager.create_payment_session(
    user_id=USER_ID,
    expiry_time_in_minutes=60,
    limits={"maxSpendAmount": {"value": "1.00", "currency": "USD"}},
    client_token=client_token(),
)
print(f"Research:     {research['paymentSessionId']} ($1.00 / 60 min)")

# 심층 분석: 더 큰 budget, 더 긴 expiry
deep = manager.create_payment_session(
    user_id=USER_ID,
    expiry_time_in_minutes=480,
    limits={"maxSpendAmount": {"value": "5.00", "currency": "USD"}},
    client_token=client_token(),
)
print(f"Deep analysis: {deep['paymentSessionId']} ($5.00 / 480 min)")

print("\nSame user, same wallet, independent budgets.")
print("The agent receives whichever sessionId matches the task.")

## 7. 사용자의 모든 Instrument 목록 표시

> **🖥️ Backend operation:** `ListPaymentInstruments`는 일반적으로 ops tooling, support dashboard 또는 애플리케이션에서 사용자에게 rendering하는 wallet-selector UI에 사용되는 backend 호출입니다.

지정된 사용자의 모든 wallet을 확인합니다. wallet 선택 UI, account dashboard 또는 support tool을 구축할 때 유용합니다. 호출 범위는 사용자별로 지정됩니다. 아래 셀에서는 Tutorial 00과 1섹션에서 생성한 두 사용자의 instrument 목록을 표시합니다.

`ListPaymentInstruments`는 전체 wallet detail이 아니라 간단한 summary(ID, type, status, timestamp)를 반환합니다. 지정한 instrument의 network 또는 wallet address가 필요하면 해당 `paymentInstrumentId`로 `GetPaymentInstrument`를 호출합니다.

In [ ]:
# 생성한 각 사용자의 모든 instrument 목록 표시
# SDK가 {'paymentInstruments': [...]}를 반환하며 ListPaymentInstruments의
# userId requirement를 SDK 내부에서 처리함
for label, user_id in [
    ("Tutorial 00 user", USER_ID),
    ("Section 1 new user", NEW_USER_ID),
]:
    resp = manager.list_payment_instruments(
        user_id=user_id,
        payment_connector_id=CONNECTOR_ID,
    )
    instruments = resp.get("paymentInstruments", [])
    print(f"\n\u2705 {label} ({user_id}): {len(instruments)} instrument(s)")
    for inst in instruments:
        print(f"  {inst['paymentInstrumentId']}")
        print(f"    type:    {inst.get('paymentInstrumentType', 'unknown')}")
        print(f"    status:  {inst.get('status', 'unknown')}")
        print(f"    created: {inst.get('createdAt', 'unknown')}")

## 8. Session의 남은 Budget 확인

> **🖥️ Backend operation:** `GetPaymentSession`은 backend 호출입니다. application이 이를 query하여 사용자에게 남은 budget을 표시하고 agent에 추가 작업을 routing할지 결정하거나 지출을 logging합니다.

`GetPaymentSession`은 실시간 남은 budget인 `availableLimits.availableSpendAmount`를 반환합니다. agent에 더 많은 작업을 routing하기 전에 backend에서 이를 호출하여 사용자에게 남은 금액을 보여줍니다. 아래 셀에서는 6섹션에서 생성한 세 session을 모두 검사하여 context에서 shape를 확인합니다.

In [ ]:
# 위에서 생성한 세 session의 남은 budget 확인
# SDK가 top level에 limits / availableLimits / expiryTimeInMinutes가 있는 flat dict 반환
for label, created in [
    ("Quick lookup", quick),
    ("Research", research),
    ("Deep analysis", deep),
]:
    sid = created["paymentSessionId"]
    sess = manager.get_payment_session(
        user_id=USER_ID,
        payment_session_id=sid,
    )
    budget = sess.get("limits", {}).get("maxSpendAmount", {})
    available = sess.get("availableLimits", {}).get("availableSpendAmount", {})
    print(f"\n{label} — {sid}")
    print(f"  Budget:    {budget.get('value', 'N/A')} {budget.get('currency', '')}")
    print(f"  Available: {available.get('value', 'N/A')} {available.get('currency', '')}")
    print(f"  Expiry:    {sess.get('expiryTimeInMinutes', 'N/A')} minutes")

## 요약

### Funding Option(Part 1 — 최종 사용자)

| 방식 | 사용 사례 | Provider |
|--------|----------|----------|
| Circle faucet | Testnet(무료) | 둘 다 |
| Direct USDC transfer | 사용자가 외부 wallet에서 전송 | 둘 다 |
| Coinbase Onramp URL | Fiat → crypto(credit card, bank) | Coinbase |
| Stripe Onramp | Fiat → crypto(credit card, bank, Apple Pay) | Privy |
| Coinbase WalletHub | 하나의 UI에서 fund + delegate(Coinbase에서 관리) | Coinbase |
| Privy reference frontend | app UI를 통해 fund + delegate(prod에서 자체 호스팅) | Privy |

### Session Pattern(Part 2 — backend)

| Pattern | Budget | Expiry | 사용 사례 |
|---------|--------|--------|----------|
| 빠른 조회 | $0.10 | 15 min | 단일 API 호출 |
| research task | $1.00 | 60 min | multi-endpoint research |
| 심층 분석 | $5.00 | 480 min | 확장된 workflow |
| budget 상한 없음 | `limits` 생략 | 60 min | 신뢰할 수 있는 internal agent |

### 지원 Network

| Network | Chain | Testnet | Faucet |
|---------|-------|---------|--------|
| ETHEREUM | Base Sepolia | `eip155:84532` | faucet.circle.com → Base Sepolia |
| SOLANA | Solana Devnet | `solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1` | faucet.circle.com → Solana Devnet |


## 검증

위 셀이 오류 없이 실행되었다면 wallet operation이 성공한 것입니다. 각 instrument의 wallet address가 출력되고 `GetPaymentInstrumentBalance`에서 0이 아닌 USDC balance가 반환되어야 합니다. 이는 wallet이 생성되고 자금이 입금되었으며 backend에서 액세스할 수 있음을 의미합니다.

## 리소스 정리

이 튜토리얼에서 생성한 payment session 세 개(quick lookup, research task, deep analysis)는 구성된 `expiryTimeInMinutes`가 지나면 자동으로 만료됩니다. 모든 payment resource를 삭제하려면 Tutorial 00의 cleanup 셀을 실행합니다.

이 Notebook에서 생성한 Instrument(`NEW_INSTRUMENT_ID`)는 Tutorial 00에서 Payment Manager를 삭제할 때 함께 정리됩니다.

# 축하합니다!

이제 onboarding(create, fund, delegate)과 그 주변에서 application이 실행하는 backend operation(balance 확인, multi-network wallet, session pattern)을 포함한 전체 wallet lifecycle을 이해했습니다.